# Qwen3.5 Small Models — CNIE Extraction Experiment

Testing **Qwen3.5 small models** (released March 2, 2026) for structured CNIE field extraction.

**Approach:** Build `llama.cpp` from source (since `llama-cpp-python` doesn't support Qwen3.5 yet),
run `llama-server` as a local OpenAI-compatible API, call it from Python.

| # | Model | GGUF Size | Notes |
|---|---|---|---|
| A | **Qwen3.5-0.8B** Q5_K_M | 590 MB | Ultra-fast baseline |
| B | **Qwen3.5-2B** Q5_K_M | 1.44 GB | Fast and capable |
| C | **Qwen3.5-4B** Q5_K_M | 3.14 GB | Best balance |
| D | **Qwen3.5-9B** Q4_K_M | 5.68 GB | Best quality, may be slow on CPU |

---
## Step 1: Build llama.cpp from source

This compiles the latest llama.cpp with Qwen3.5 DeltaNet support.
Takes ~2-3 minutes. Only need to run once per session.

In [ ]:
%%bash
# Install build dependencies
apt-get update -qq
apt-get install -y -qq build-essential cmake curl libcurl4-openssl-dev > /dev/null 2>&1
echo "Build tools installed."

# Clone llama.cpp (latest)
if [ ! -d "llama.cpp" ]; then
    git clone --depth 1 https://github.com/ggml-org/llama.cpp
    echo "Cloned llama.cpp"
else
    echo "llama.cpp already cloned"
fi

# Build CPU-only (works on all Colab instances)
cmake llama.cpp -B llama.cpp/build \
    -DBUILD_SHARED_LIBS=OFF \
    -DGGML_CUDA=OFF \
    -DCMAKE_BUILD_TYPE=Release

cmake --build llama.cpp/build --config Release -j$(nproc) \
    --target llama-cli llama-server

# Copy binaries to llama.cpp root for easy access
cp llama.cpp/build/bin/llama-cli llama.cpp/
cp llama.cpp/build/bin/llama-server llama.cpp/

echo ""
echo "============================================"
echo "llama.cpp built successfully!"
ls -lh llama.cpp/llama-cli llama.cpp/llama-server
echo "============================================"

---
## Step 2: Download Model

Uncomment ONE model. Start with **Model C (4B)**.

In [ ]:
import os, time

# =============================================
# MODEL A: Qwen3.5-0.8B Q5_K_M (590 MB)
# =============================================
# MODEL_REPO = "unsloth/Qwen3.5-0.8B-GGUF"
# MODEL_FILE = "Qwen3.5-0.8B-Q5_K_M.gguf"
# MODEL_LABEL = "Qwen3.5-0.8B Q5_K_M"

# =============================================
# MODEL B: Qwen3.5-2B Q5_K_M (1.44 GB)
# =============================================
# MODEL_REPO = "unsloth/Qwen3.5-2B-GGUF"
# MODEL_FILE = "Qwen3.5-2B-Q5_K_M.gguf"
# MODEL_LABEL = "Qwen3.5-2B Q5_K_M"

# =============================================
# MODEL C: Qwen3.5-4B Q5_K_M (3.14 GB) — START HERE
# =============================================
MODEL_REPO = "unsloth/Qwen3.5-4B-GGUF"
MODEL_FILE = "Qwen3.5-4B-Q5_K_M.gguf"
MODEL_LABEL = "Qwen3.5-4B Q5_K_M"

# =============================================
# MODEL D: Qwen3.5-9B Q4_K_M (5.68 GB)
# =============================================
# MODEL_REPO = "unsloth/Qwen3.5-9B-GGUF"
# MODEL_FILE = "Qwen3.5-9B-Q4_K_M.gguf"
# MODEL_LABEL = "Qwen3.5-9B Q4_K_M"

# Download
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download

print(f"Downloading {MODEL_LABEL}...")
t0 = time.time()
model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print(f"Downloaded in {time.time()-t0:.1f}s")
print(f"Path: {model_path}")
print(f"Size: {os.path.getsize(model_path) / 1e9:.2f} GB")

---
## Step 3: Start llama-server

Launches an OpenAI-compatible API server in the background on port 8080.

In [ ]:
import subprocess, requests

SERVER_PORT = 8080
N_THREADS = os.cpu_count() or 4

# Kill any existing server
!pkill -f llama-server 2>/dev/null || true
time.sleep(1)

# Start llama-server in background
cmd = [
    "./llama.cpp/llama-server",
    "--model", model_path,
    "--ctx-size", "2048",
    "--threads", str(N_THREADS),
    "--port", str(SERVER_PORT),
    "--chat-template-kwargs", '{"enable_thinking":false}',
    "--temp", "0.1",
    "--log-disable",
]

print(f"Starting llama-server on port {SERVER_PORT}...")
print(f"Model: {MODEL_LABEL} | Threads: {N_THREADS}")

server_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for server to be ready
print("Waiting for server to load model...", end="")
for i in range(120):  # up to 2 minutes
    time.sleep(1)
    try:
        r = requests.get(f"http://localhost:{SERVER_PORT}/health", timeout=2)
        if r.status_code == 200:
            print(f" ready! ({i+1}s)")
            break
    except:
        print(".", end="", flush=True)
else:
    print("\nERROR: Server did not start. Check stderr:")
    print(server_proc.stderr.read().decode()[:2000])

In [ ]:
# Quick health check
r = requests.get(f"http://localhost:{SERVER_PORT}/health")
print(f"Server status: {r.json()}")

---
## Step 4: System Prompt + Extraction Function

In [ ]:
import json, re

API_URL = f"http://localhost:{SERVER_PORT}/v1/chat/completions"

# ============================================================
# ENGINEERED SYSTEM PROMPT — built from real OCR error analysis
# ============================================================
SYSTEM_PROMPT = """You extract personal data from Moroccan ID card (CNIE) OCR output.

INPUT: A list of text detections from OCR on a CNIE card. Each has text and confidence (0-1). The OCR reads BOTH Arabic and French from the same image.

TEMPLATE TEXT — IGNORE THESE (not personal data, never use as name/place):
- ROYAUME DU MAROC, CARTE NATIONALE D'IDENTITE
- المملكة, المغربية, المعربية, البطاقة الوطنية, للتعريف
- Né le, مزداد بتاريخ, مزداد بتانيخ (prefix before birth date)
- Valable jusqu'au, صالحة الى غاية (prefix before expiry)
- المدير العام للأمن الوطني, المدير العام للإمن الوطنى, عبد اللطيف حموشي, الوطني (signature)

HOW TO IDENTIFY PERSONAL NAMES:
- Names appear as Arabic+French PAIRS: one Arabic word + one UPPERCASE French word
- Example: شافي is paired with CHAFI → last name. بلال is paired with BILAL → first name.
- French names are ALWAYS UPPERCASE on the card. Return them UPPERCASE.
- If you see an Arabic word next to an UPPERCASE French word, they are the same name.
- المعربية, المملكة, البطاقة, الوطنية, للتعريف are NEVER names — they are template headers.

OTHER FIELDS:
- Birth date: DD.MM.YYYY number near "Né le" / "مزداد بتاريخ"
- Birth place: French city after "à/a" (e.g. "a RABAT"), Arabic city name nearby
- Expiry date: DD.MM.YYYY number near "Valable jusqu'au" / "صالحة الى غاية"
- Card number: 2 letters + 5-6 digits (e.g. AB123456, AS13538)
- Gender: single letter M or F

KNOWN OCR ERRORS IN ARABIC — the model confuses similar characters:
- ر↔ن (dot position), غ↔ع (dot above), ب↔ت↔ث (dot count), ة↔ط
- Example: الرياة is actually الرباط (Rabat), المعربية is actually المغربية
- When Arabic text seems garbled, use the French text as ground truth

CROSS-REFERENCING:
- French Latin text is MORE RELIABLE than Arabic for names and places
- If Arabic says الرياة but French says RABAT → birth_place_ar = الرباط
- Dates and card numbers are equally reliable in both scripts

CONFIDENCE: Ignore detections with confidence < 0.5 (likely garbage).

OUTPUT — return ONLY this JSON, no explanation:
{"last_name_fr": "UPPERCASE", "first_name_fr": "UPPERCASE", "last_name_ar": "", "first_name_ar": "", "birth_date": "DD.MM.YYYY", "birth_place_fr": "UPPERCASE", "birth_place_ar": "", "card_number": "", "expiry_date": "DD.MM.YYYY", "gender": "M or F"}
Set any field not found to null. Do NOT include thinking or reasoning."""


def parse_json_from_response(raw):
    """Extract JSON from model response, handling thinking tags and extra text."""
    cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    match = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {"_raw": raw, "_error": "json_parse_failed"}


def score_extraction(extracted, expected):
    """Compare extracted fields to expected. Returns (correct, total, details)."""
    correct = 0
    total = len(expected)
    details = []
    for field, exp_val in expected.items():
        got = extracted.get(field)
        exp_norm = str(exp_val).strip().upper() if exp_val else None
        got_norm = str(got).strip().upper() if got else None
        match = exp_norm == got_norm
        if match:
            correct += 1
            details.append(f"  OK   {field}: '{got}'")
        else:
            details.append(f"  MISS {field}: expected '{exp_val}' got '{got}'")
    return correct, total, details


def extract_fields(ocr_text, system_prompt=SYSTEM_PROMPT, verbose=True):
    """Call llama-server API and extract structured fields."""
    t0 = time.time()

    payload = {
        "model": "qwen",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Extract fields from this CNIE OCR output:\n\n{ocr_text}"},
        ],
        "max_tokens": 512,
        "temperature": 0.1,
        "response_format": {"type": "json_object"},
    }

    r = requests.post(API_URL, json=payload, timeout=300)
    r.raise_for_status()
    data = r.json()

    elapsed = time.time() - t0
    raw = data["choices"][0]["message"]["content"]
    usage = data.get("usage", {})

    if verbose:
        print(f"Model: {MODEL_LABEL}")
        print(f"Inference time: {elapsed:.2f}s")
        print(f"Tokens — prompt: {usage.get('prompt_tokens', '?')}, "
              f"completion: {usage.get('completion_tokens', '?')}")

    parsed = parse_json_from_response(raw)
    return {"fields": parsed, "time_s": round(elapsed, 2), "model": MODEL_LABEL, "raw": raw}


print(f"Engineered prompt + extract_fields() + score_extraction() ready — calling {API_URL}")

---
## Step 5: Test Samples

In [ ]:
# ============================================================
# SAMPLE 1: Clean OCR
# ============================================================
SAMPLE_OCR_1 = """
Arabic OCR:
  - text: 'المملكة المغربية', confidence: 0.95
  - text: 'بطاقة التعريف الوطنية', confidence: 0.92
  - text: 'الشافعي', confidence: 0.88
  - text: 'بلال', confidence: 0.91
  - text: 'المزداد بتاريخ', confidence: 0.85
  - text: 'بالرباط', confidence: 0.90

French OCR:
  - text: 'ROYAUME DU MAROC', confidence: 0.97
  - text: 'CHAFI', confidence: 0.93
  - text: 'BILAL', confidence: 0.95
  - text: 'Ne le 22.01.2007', confidence: 0.89
  - text: 'a RABAT', confidence: 0.92
  - text: 'Valable jusqu au 19.03.2029', confidence: 0.90
  - text: 'AB123456', confidence: 0.94
  - text: 'M', confidence: 0.96
"""

EXPECTED_1 = {
    "last_name_fr": "CHAFI",
    "first_name_fr": "BILAL",
    "last_name_ar": "الشافعي",
    "first_name_ar": "بلال",
    "birth_date": "22.01.2007",
    "birth_place_fr": "RABAT",
    "birth_place_ar": "الرباط",
    "card_number": "AB123456",
    "expiry_date": "19.03.2029",
    "gender": "M",
}

print("=" * 60)
print(f"TEST 1 — Clean OCR — {MODEL_LABEL}")
print("=" * 60)
result1 = extract_fields(SAMPLE_OCR_1)
print("\nExtracted:")
print(json.dumps(result1["fields"], indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# SAMPLE 2: Noisy OCR (0/O, 1/I, 7/T swaps)
# ============================================================
SAMPLE_OCR_2 = """
Arabic OCR:
  - text: 'الملكة المكربية', confidence: 0.72
  - text: 'بطاقة التعريف الوطنيى', confidence: 0.68
  - text: 'لوح', confidence: 0.55
  - text: 'اجا', confidence: 0.60

French OCR:
  - text: 'R0YAUME DU MAR0C', confidence: 0.75
  - text: 'CHAF1', confidence: 0.70
  - text: 'B1LAL', confidence: 0.72
  - text: 'Ne le 22.O1.20O7', confidence: 0.65
  - text: 'a RABA7', confidence: 0.60
  - text: 'Va1ab1e jusqu au 19.03.2029', confidence: 0.58
  - text: 'A8123456', confidence: 0.55
"""

print("=" * 60)
print(f"TEST 2 — Noisy OCR — {MODEL_LABEL}")
print("=" * 60)
result2 = extract_fields(SAMPLE_OCR_2)
print("\nExtracted:")
print(json.dumps(result2["fields"], indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# SAMPLE 3: REAL PaddleOCR output (from our API, single Arabic model)
# ============================================================
REAL_OCR = """
OCR detections:
  - text: 'المعربية', confidence: 0.93
  - text: 'المملكة', confidence: 0.93
  - text: 'ROYAUME DU MAROC', confidence: 0.99
  - text: 'للتعريف', confidence: 0.96
  - text: 'البطاقة الوطنية', confidence: 0.97
  - text: 'CARTE NATIONALE D IDENTITE', confidence: 0.99
  - text: 'بلال', confidence: 0.95
  - text: 'BILAL', confidence: 0.99
  - text: 'شافي', confidence: 0.99
  - text: 'CHAFI', confidence: 0.99
  - text: 'Né le', confidence: 0.87
  - text: '22.01.2001', confidence: 0.92
  - text: 'مزداد بتانيخ', confidence: 0.88
  - text: 'الرياة', confidence: 0.71
  - text: 'a RABAT', confidence: 0.90
  - text: 'Valable jusqu au', confidence: 0.99
  - text: '19.03.2029', confidence: 0.99
  - text: 'صالحة الى غاية', confidence: 0.97
  - text: 'AS13538', confidence: 0.99
  - text: 'M', confidence: 0.99
  - text: 'المدير العام للإمن الوطنى', confidence: 0.97
  - text: 'Thy', confidence: 0.53
  - text: 'عبد اللطيّف حموشي', confidence: 0.91
"""

EXPECTED_REAL = {
    "last_name_fr": "CHAFI",
    "first_name_fr": "BILAL",
    "last_name_ar": "شافي",
    "first_name_ar": "بلال",
    "birth_date": "22.01.2001",
    "birth_place_fr": "RABAT",
    "birth_place_ar": "الرباط",
    "card_number": "AS13538",
    "expiry_date": "19.03.2029",
    "gender": "M",
}

print("=" * 60)
print(f"TEST 3 — REAL OCR — {MODEL_LABEL}")
print("=" * 60)
result3 = extract_fields(REAL_OCR)
print("\nExtracted:")
print(json.dumps(result3["fields"], indent=2, ensure_ascii=False))

print(f"\nSCORECARD — Real OCR:")
print("-" * 60)
correct, total, details = score_extraction(result3["fields"], EXPECTED_REAL)
for d in details:
    print(d)
print(f"\nScore: {correct}/{total} fields correct | Time: {result3['time_s']}s")

---
## Step 6: Auto-Score

In [ ]:
def score_extraction(extracted, expected):
    """Compare extracted fields to expected."""
    correct = 0
    total = len(expected)
    details = []
    for field, exp_val in expected.items():
        got = extracted.get(field)
        exp_norm = str(exp_val).strip().upper() if exp_val else None
        got_norm = str(got).strip().upper() if got else None
        match = exp_norm == got_norm
        if match:
            correct += 1
            details.append(f"  OK   {field}: '{got}'")
        else:
            details.append(f"  MISS {field}: expected '{exp_val}' got '{got}'")
    return correct, total, details


print(f"\nSCORECARD — {MODEL_LABEL} — Sample 1 (clean OCR):")
print("-" * 60)
correct, total, details = score_extraction(result1["fields"], EXPECTED_1)
for d in details:
    print(d)
print(f"\nScore: {correct}/{total} fields correct | Time: {result1['time_s']}s")

---
## Step 7: Few-Shot Prompt Test

In [ ]:
SYSTEM_PROMPT_FEWSHOT = """You are a Moroccan ID card (CNIE) data extraction assistant.
You receive raw OCR text from a CNIE card in Arabic and French.
Extract the fields and return ONLY valid JSON.

Example input:
Arabic OCR:
  - text: 'العلوي', confidence: 0.90
  - text: 'محمد', confidence: 0.88
French OCR:
  - text: 'ALAOUI', confidence: 0.93
  - text: 'MOHAMMED', confidence: 0.91
  - text: 'Ne le 15.06.1990', confidence: 0.89
  - text: 'a CASABLANCA', confidence: 0.85
  - text: 'Valable jusqu au 20.06.2030', confidence: 0.87
  - text: 'CD987654', confidence: 0.92
  - text: 'M', confidence: 0.95

Example output:
{"last_name_fr": "ALAOUI", "first_name_fr": "MOHAMMED", "last_name_ar": "العلوي", "first_name_ar": "محمد", "birth_date": "15.06.1990", "birth_place_fr": "CASABLANCA", "birth_place_ar": null, "card_number": "CD987654", "expiry_date": "20.06.2030", "gender": "M"}

Rules:
- If a field is not found, set it to null
- Fix obvious OCR errors (e.g. 0 instead of O in names, 1 instead of I)
- Return ONLY the JSON object, no explanation
"""

print("=" * 60)
print(f"FEW-SHOT — {MODEL_LABEL} — Sample 1 (clean)")
print("=" * 60)
result_fs1 = extract_fields(SAMPLE_OCR_1, system_prompt=SYSTEM_PROMPT_FEWSHOT)
print("\nFew-shot extracted:")
print(json.dumps(result_fs1["fields"], indent=2, ensure_ascii=False))

print("\n" + "=" * 60)
print(f"FEW-SHOT — {MODEL_LABEL} — Sample 2 (noisy)")
print("=" * 60)
result_fs2 = extract_fields(SAMPLE_OCR_2, system_prompt=SYSTEM_PROMPT_FEWSHOT)
print("\nFew-shot extracted:")
print(json.dumps(result_fs2["fields"], indent=2, ensure_ascii=False))

In [ ]:
# Score comparison
print("ZERO-SHOT vs FEW-SHOT (Sample 1 — clean):")
print("=" * 60)
c_zero, t, _ = score_extraction(result1["fields"], EXPECTED_1)
c_few, _, _ = score_extraction(result_fs1["fields"], EXPECTED_1)
print(f"  Zero-shot: {c_zero}/{t} correct | {result1['time_s']}s")
print(f"  Few-shot:  {c_few}/{t} correct | {result_fs1['time_s']}s")
print(f"  Winner:    {'Few-shot' if c_few > c_zero else 'Zero-shot' if c_zero > c_few else 'Tie'}")

---
## Step 8: Comparison Table

Fill in after testing each model. To switch:
1. Change uncommented model in Step 2
2. Re-run Step 2 (download)
3. Re-run Step 3 (restart server with new model)
4. Re-run tests

In [ ]:
comparison = {
    "Qwen3.5-0.8B": {
        "size_gb": 0.59,
        "time_clean_s": None,
        "time_noisy_s": None,
        "score_clean": None,
        "score_noisy": None,
        "json_valid": None,
        "arabic_ok": None,
        "prompt": "",
        "notes": "",
    },
    "Qwen3.5-2B": {
        "size_gb": 1.44,
        "time_clean_s": None,
        "time_noisy_s": None,
        "score_clean": None,
        "score_noisy": None,
        "json_valid": None,
        "arabic_ok": None,
        "prompt": "",
        "notes": "",
    },
    "Qwen3.5-4B": {
        "size_gb": 3.14,
        "time_clean_s": None,
        "time_noisy_s": None,
        "score_clean": None,
        "score_noisy": None,
        "json_valid": None,
        "arabic_ok": None,
        "prompt": "",
        "notes": "",
    },
    "Qwen3.5-9B": {
        "size_gb": 5.68,
        "time_clean_s": None,
        "time_noisy_s": None,
        "score_clean": None,
        "score_noisy": None,
        "json_valid": None,
        "arabic_ok": None,
        "prompt": "",
        "notes": "",
    },
}

print(f"{'Model':<16} {'Size':<8} {'Clean(s)':<10} {'Noisy(s)':<10} {'Clean':<8} {'Noisy':<8} {'JSON':<7} {'Arabic':<8} {'Prompt':<7} {'Notes'}")
print("=" * 115)
for model, d in comparison.items():
    print(f"{model:<16} {str(d['size_gb'])+'GB':<8} {str(d['time_clean_s'] or '—'):<10} {str(d['time_noisy_s'] or '—'):<10} "
          f"{str(d['score_clean'] or '—'):<8} {str(d['score_noisy'] or '—'):<8} {str(d['json_valid'] or '—'):<7} "
          f"{str(d['arabic_ok'] or '—'):<8} {d['prompt'] or '—':<7} {d['notes']}")

---
## Step 9: Bridge Function (for Flask integration)

In [ ]:
def format_ocr_for_llm(arabic_results, french_results):
    """Format PaddleOCR output for the LLM prompt."""
    lines = ["Arabic OCR:"]
    for d in arabic_results:
        lines.append(f"  - text: '{d['text']}', confidence: {d['confidence']}")
    lines.append("")
    lines.append("French OCR:")
    for d in french_results:
        lines.append(f"  - text: '{d['text']}', confidence: {d['confidence']}")
    return "\n".join(lines)

# Quick test
mock_ar = [{"text": "الشافعي", "confidence": 0.88}, {"text": "بلال", "confidence": 0.91}]
mock_fr = [{"text": "CHAFI", "confidence": 0.93}, {"text": "BILAL", "confidence": 0.95}]
print("Bridge function output:")
print(format_ocr_for_llm(mock_ar, mock_fr))

---
## Step 10: Cleanup — Stop Server

In [ ]:
# Run this when done or before switching models
!pkill -f llama-server 2>/dev/null && echo "Server stopped." || echo "No server running."

---
## Decision Checklist

After testing all 4 models:

1. **Winner?** Which model + prompt gave best score?
2. **Speed?** Under 10s on Colab CPU?
3. **Arabic?** Correctly extracted Arabic names?
4. **JSON reliable?** Always valid parseable JSON?
5. **Worth 9B?** Much better than 4B to justify speed cost?

Share results and we integrate the winner into Flask.

**For Flask deployment:** We'll use `llama-server` as a sidecar process,
or switch to `llama-cpp-python` once it adds Qwen3.5 support.